In [2]:
import torch, nltk, pickle
from torch import nn
from collections import Counter
from transformers import BatchEncoding, PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutput

from torch.utils.data import DataLoader
import numpy as np
import sys, time, os

# Part 1. Tokenization.

In [3]:
def lowercase_tokenizer(text):
    return [t.lower() for t in nltk.word_tokenize(text)]

In [4]:
import nltk
nltk.download('punkt_tab')
lowercase_tokenizer("Alpha is Beta")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /cephyr/users/yingshuo/Alvis/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


['alpha', 'is', 'beta']

In [5]:
def build_tokenizer(train_file, tokenize_fun=lowercase_tokenizer, max_voc_size=None, model_max_length=None,
                    pad_token='<PAD>', unk_token='<UNK>', bos_token='<BOS>', eos_token='<EOS>'):
    """ Build a tokenizer from the given file.

        Args:
             train_file:        The name of the file containing the training texts.
             tokenize_fun:      The function that maps a text to a list of string tokens.
             max_voc_size:      The maximally allowed size of the vocabulary.
             model_max_length:  Truncate texts longer than this length.
             pad_token:         The dummy string corresponding to padding.
             unk_token:         The dummy string corresponding to out-of-vocabulary tokens.
             bos_token:         The dummy string corresponding to the beginning of the text.
             eos_token:         The dummy string corresponding to the end the text.
    """

    # TODO: build the vocabulary, possibly truncating it to max_voc_size if that is specified.
    # Then return a tokenizer object (implemented below).
    ...
    counter = Counter()

    with open(train_file, encoding='utf-8') as file:
        for paragraph in file:
            tokens = tokenize_fun(paragraph.strip())
            counter.update(tokens)

    words = []
    for word, count in counter.most_common():
        words.append(word)
    
    if max_voc_size is not None:
        words = words[:max_voc_size - 4]  # dummy tokens

    vocabulary = [pad_token, unk_token, bos_token, eos_token] + words

    return A1Tokenizer(
        vocabulary=vocabulary,
        tokenize_fun=tokenize_fun,
        model_max_length=model_max_length,
        pad_token=pad_token,
        unk_token=unk_token,
        bos_token=bos_token,
        eos_token=eos_token
    )

In [6]:
class A1Tokenizer:
    """A minimal implementation of a tokenizer similar to tokenizers in the HuggingFace library."""

    def __init__(self, vocabulary, tokenize_fun, model_max_length=None,
                 pad_token='<PAD>', unk_token='<UNK>',
                 bos_token='<BOS>', eos_token='<EOS>'):
        # TODO: store all values you need in order to implement __call__ below.
        self.vocabulary = vocabulary
        
        self.token2id = {}
        for i, tok in enumerate(vocabulary):
            self.token2id[tok] = i
            
        self.id2token = {}
        for tok, i in self.token2id.items():
            self.id2token[i] = tok

        self.tokenize_fun = tokenize_fun
        self.model_max_length = model_max_length

        self.pad_token = pad_token
        self.unk_token = unk_token
        self.bos_token = bos_token
        self.eos_token = eos_token

        self.pad_token_id = self.token2id[pad_token]
        self.unk_token_id = self.token2id[unk_token]
        self.bos_token_id = self.token2id[bos_token]
        self.eos_token_id = self.token2id[eos_token]

    def __call__(self, texts, truncation=False, padding=False, return_tensors=None):
        """Tokenize the given texts and return a BatchEncoding containing the integer-encoded tokens.
           
           Args:
             texts:           The texts to tokenize.
             truncation:      Whether the texts should be truncated to model_max_length.
             padding:         Whether the tokenized texts should be padded on the right side.
             return_tensors:  If None, then return lists; if 'pt', then return PyTorch tensors.

           Returns:
             A BatchEncoding where the field `input_ids` stores the integer-encoded texts.
        """
        if return_tensors and return_tensors != 'pt':
            raise ValueError('Should be pt')
        
        # TODO: Your work here is to split the texts into words and map them to integer values.
        # 
        # - If `truncation` is set to True, the length of the encoded sequences should be 
        #   at most self.model_max_length.
        # - If `padding` is set to True, then all the integer-encoded sequences should be of the
        #   same length. That is: the shorter sequences should be "padded" by adding dummy padding
        #   tokens on the right side.
        # - If `return_tensors` is undefined, then the returned `input_ids` should be a list of lists.
        #   Otherwise, if `return_tensors` is 'pt', then `input_ids` should be a PyTorch 2D tensor.

        # TODO: Return a BatchEncoding where input_ids stores the result of the integer encoding.
        # Optionally, if you want to be 100% HuggingFace-compatible, you should also include an 
        # attention mask of the same shape as input_ids. In this mask, padding tokens correspond
        # to the the value 0 and real tokens to the value 1.

        encoded_sequences = []
        for text in texts:
            tokens = [self.bos_token] + self.tokenize_fun(text) + [self.eos_token]

            if truncation and self.model_max_length is not None:
                tokens = tokens[:self.model_max_length]
    
            ids = []
            for tok in tokens:
                if tok in self.token2id:
                    ids.append(self.token2id[tok])
                else:
                    ids.append(self.unk_token_id)

            encoded_sequences.append(ids)

        attention_mask = [[1] * len(seq) for seq in encoded_sequences]
        
        if padding:
            max_len = 0
            for seq in encoded_sequences:
                if len(seq) > max_len:
                    max_len = len(seq)

            padded_sequences = []
            padded_masks = []

            for seq, mask in zip(encoded_sequences, attention_mask):
                pad_len = max_len - len(seq)
                padded_sequences.append(seq + [self.pad_token_id] * pad_len)
                padded_masks.append(mask + [0] * pad_len)

            encoded_sequences = padded_sequences
            attention_mask = padded_masks

        if return_tensors == 'pt':
            encoded_sequences = torch.tensor(encoded_sequences, dtype=torch.long)
            attention_mask = torch.tensor(attention_mask, dtype=torch.long)

        return BatchEncoding({'input_ids': encoded_sequences, 'attention_mask': attention_mask})

    def __len__(self):
        """Return the size of the vocabulary."""
        return len(self.vocabulary)
    
    def save(self, filename):
        """Save the tokenizer to the given file."""
        with open(filename, 'wb') as f:
            pickle.dump(self, f)

    @staticmethod
    def from_file(filename):
        """Load a tokenizer from the given file."""
        with open(filename, 'rb') as f:
            return pickle.load(f)

## 1.2 Sanity Check

In [7]:
max_voc_size = 10000

tokenizer = build_tokenizer(
    train_file="train.txt",
    max_voc_size=max_voc_size,
    model_max_length=128
)

assert len(tokenizer) <= max_voc_size
print("Sanity check 1.2.1： PASSED")

Sanity check 1.2.1： PASSED


In [8]:
special_tokens = [
    tokenizer.pad_token,
    tokenizer.unk_token,
    tokenizer.bos_token,
    tokenizer.eos_token
]

real_words = tokenizer.vocabulary[4:]

for token in special_tokens:
    assert token not in real_words

print("Sanity check 1.2.2： PASSED")

Sanity check 1.2.2： PASSED


In [9]:
for word in ["the", "and"]:
    assert word in tokenizer.token2id

for word in ["cuboidal", "epiglottis"]:
    assert word not in tokenizer.token2id

print("Sanity check 1.2.3： PASSED")

Sanity check 1.2.3： PASSED


In [10]:
test_word = "the"

word_id = tokenizer.token2id[test_word]
word_back = tokenizer.id2token[word_id]

assert word_back == test_word
print("Sanity check 1.2.4： PASSED")

Sanity check 1.2.4： PASSED


## 1.3 Sanity Check

In [11]:
test_texts = [
    "This is a test.",
    "Another test."
]

encoded = tokenizer(
    test_texts,
    return_tensors="pt",
    padding=True,
    truncation=True
)

# print(encoded)

tokenizer.save("my_tokenizer.pkl")
loaded_tokenizer = A1Tokenizer.from_file("my_tokenizer.pkl")
loaded_encoded = loaded_tokenizer(
    test_texts,
    return_tensors="pt",
    padding=True,
    truncation=True
)

assert torch.equal(encoded["input_ids"], loaded_encoded["input_ids"])
print("Sanity check 1.3： PASSED")

Sanity check 1.3： PASSED


# Part 2: Loading the text files and creating batches

In [12]:
from datasets import load_dataset

TRAIN_FILE = "train.txt"
VAL_FILE = "val.txt"

dataset = load_dataset(
    "text",
    data_files={
        "train": TRAIN_FILE,
        "val": VAL_FILE
    }
)

dataset = dataset.filter(lambda x: x["text"].strip() != "")

print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["val"]))

Train size: 147059
Validation size: 17874


## 2.1 Sanity Check

In [13]:
assert len(dataset["train"]) > 140000
assert len(dataset["val"]) > 17000

print("Sanity check 2.1: PASSED")

Sanity check 2.1: PASSED


In [14]:
bs = 4
dl = DataLoader(
    dataset["train"],
    batch_size=bs,
    shuffle=True
)

for batch in dl:
    # print(batch)
    break
    
assert len(batch["text"]) == bs

print("Sanity check 2.2: PASSED")

Sanity check 2.2: PASSED


# Part 3. Defining the model

In [15]:
class A1RNNModelConfig(PretrainedConfig):
    """Configuration object that stores hyperparameters that define the RNN-based language model."""
    def __init__(self, vocab_size, embedding_size, hidden_size, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.embedding_size = embedding_size

class A1RNNModel(PreTrainedModel):
    """The neural network model that implements a RNN-based language model."""
    config_class = A1RNNModelConfig
    
    def __init__(self, config):
        super().__init__(config)

        self.embedding = nn.Embedding(config.vocab_size, config.embedding_size)   # (batch_size, seq_len) -> (batch_size, seq_len, embedding_size)
    
        self.rnn = nn.RNN(                                                        # -> (batch_size, seq_len, hidden_size)
            input_size=config.embedding_size,
            hidden_size=config.hidden_size,
            batch_first=True
        )
    
        self.unembedding = nn.Linear(config.hidden_size, config.vocab_size)       # -> (batch_size, seq_len, vocab_size)

        # Note: -100 is the value HuggingFace conventionally uses to refer to tokens
        # where we do not want to compute the loss.
        self.loss_func = torch.nn.CrossEntropyLoss(ignore_index=-100)


    def forward(self, input_ids, labels=None):
        """The forward pass of the RNN-based language model.
        
           Args:
             - input_ids:  The input tensor (2D), consisting of a batch of integer-encoded texts.
             - labels:     The reference tensor (2D), consisting of a batch of integer-encoded texts.
           Returns:
             A CausalLMOutput containing
               - logits:   The output tensor (3D), consisting of logits for all token positions for all vocabulary items.
               - loss:     The loss computed on this batch.               
        """
        embedded = self.embedding(input_ids)
        rnn_out, _ = self.rnn(embedded)
        logits = self.unembedding(rnn_out)

        loss = None
        if labels is not None:
            logits_pre = logits[:, :-1, :]
            labels_next = labels[:, 1:]

            loss = self.loss_func(
                logits_pre.reshape(-1, self.config.vocab_size),
                labels_next.reshape(-1)
            )

        return CausalLMOutput(logits=logits, loss=loss)

## 3.2 Sanity Check

In [16]:
V = len(tokenizer)
N = 8

config = A1RNNModelConfig(
    vocab_size=V,
    embedding_size=64,
    hidden_size=128
)

model = A1RNNModel(config)

input_ids = torch.randint(
    low=0,
    high=V,
    size=(1, N)
)
# print("input_ids shape:", input_ids.shape)

output = model(input_ids)
# print("logits shape:", output.logits.shape)
assert output.logits.shape == (1, N, V)

output_with_loss = model(input_ids, labels=input_ids)
# print("loss:", output_with_loss.loss)
assert output_with_loss.loss is not None

print("Sanity check 3.2: PASSED")

Sanity check 3.2: PASSED


# Part 4. Training the language model

In [32]:
# Hint: the following TrainingArguments hyperparameters may be relevant for your implementation:
#
# - optim:            What optimizer to use. You can assume that this is set to 'adamw_torch',
#                     meaning that we use the PyTorch AdamW optimizer.
# - eval_strategy:    You can assume that this is set to 'epoch', meaning that the model should
#                     be evaluated on the validation set after each epoch
# - use_cpu:          Force the trainer to use the CPU; otherwise, CUDA or MPS should be used.
#                     (In your code, you can just use the provided method select_device.)
# - learning_rate:    The optimizer's learning rate.
# - num_train_epochs: The number of epochs to use in the training loop.
# - per_device_train_batch_size: 
#                     The batch size to use while training.
# - per_device_eval_batch_size:
#                     The batch size to use while evaluating.
# - output_dir:       The directory where the trained model will be saved.

class A1Trainer:
    """A minimal implementation similar to a Trainer from the HuggingFace library."""

    def __init__(self, model, args, train_dataset, eval_dataset, tokenizer):
        """Set up the trainer.
           
           Args:
             model:          The model to train.
             args:           The training parameters stored in a TrainingArguments object.
             train_dataset:  The dataset containing the training documents.
             eval_dataset:   The dataset containing the validation documents.
             eval_dataset:   The dataset containing the validation documents.
             tokenizer:      The tokenizer.
        """
        self.model = model
        self.args = args
        self.train_dataset = train_dataset
        self.eval_dataset = eval_dataset
        self.tokenizer = tokenizer

        assert(args.optim == 'adamw_torch')
        assert(args.eval_strategy == 'epoch')

    def select_device(self):
        """Return the device to use for training, depending on the training arguments and the available backends."""
        if self.args.use_cpu:
            return torch.device('cpu')
        if not self.args.no_cuda and torch.cuda.is_available():
            return torch.device('cuda')
        if torch.mps.is_available():
            return torch.device('mps')
        return torch.device('cpu')
            
    def train(self):
        """Train the model."""
        args = self.args

        device = self.select_device()
        print('Device:', device)
        self.model.to(device)
        
        loss_func = torch.nn.CrossEntropyLoss(ignore_index=self.tokenizer.pad_token_id)

        # TODO: Relevant arguments: at least args.learning_rate, but you can optionally also consider
        # other Adam-related hyperparameters here.
        optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=args.learning_rate
        )

        # TODO: Relevant arguments: args.per_device_train_batch_size, args.per_device_eval_batch_size
        train_loader = DataLoader(
            self.train_dataset,
            batch_size=args.per_device_train_batch_size,
            shuffle=True
        )

        val_loader = DataLoader(
            self.eval_dataset,
            batch_size=args.per_device_eval_batch_size,
        )
        
        # TODO: Your work here is to implement the training loop.
        #       
        # for each training epoch (use args.num_train_epochs here):
        #   for each batch B in the training set:
        #
        #       PREPROCESSING AND FORWARD PASS:
        #       input_ids = apply your tokenizer to B
        #       labels = input_ids with padding replaced by -100
	    #       put input_ids and labels onto the GPU (or whatever device you use)
        #       apply the model to input_ids and labels
        #       get the loss from the model output
        #
        #       BACKWARD PASS AND MODEL UPDATE:
        #       optimizer.zero_grad()
        #       loss.backward()
        #       optimizer.step()

        loss_fct = torch.nn.CrossEntropyLoss(
            ignore_index=-100,
            reduction="sum"
        )

        for epoch in range(int(args.num_train_epochs)):
            self.model.train()
    
            total_train_loss = 0.0
            num_train_batches = 0
    
            for batch in train_loader:
                encoded = self.tokenizer(
                    batch["text"],
                    return_tensors="pt",
                    padding=True,
                    truncation=True
                )
    
                input_ids = encoded["input_ids"].to(device)
    
                labels = input_ids.clone()
                labels[labels == self.tokenizer.pad_token_id] = -100
    
                output = self.model(
                    input_ids=input_ids,
                    labels=labels
                )
    
                loss = output.loss
    
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
    
                total_train_loss += loss.item()
                num_train_batches += 1
    
            avg_train_loss = total_train_loss / num_train_batches
    

            self.model.eval()
    
            total_val_nll = 0.0
            total_val_tokens = 0
    
            with torch.no_grad():
                for batch in val_loader:
                    encoded = self.tokenizer(
                        batch["text"],
                        return_tensors="pt",
                        padding=True,
                        truncation=True
                    )
    
                    input_ids = encoded["input_ids"].to(device)
    
                    labels = input_ids.clone()
                    labels[labels == self.tokenizer.pad_token_id] = -100
    
                    output = self.model(input_ids=input_ids)
                    logits = output.logits
    
                    # For language modeling, prediction at position t
                    # should be compared with token at position t+1.
                    shift_logits = logits[:, :-1, :].contiguous()
                    shift_labels = labels[:, 1:].contiguous()
    
                    val_loss_sum = loss_fct(
                        shift_logits.view(-1, shift_logits.size(-1)),
                        shift_labels.view(-1)
                    )
    
                    num_tokens = (shift_labels != -100).sum().item()
    
                    total_val_nll += val_loss_sum.item()
                    total_val_tokens += num_tokens
    
            avg_val_loss = total_val_nll / total_val_tokens
            perplexity = np.exp(avg_val_loss)
    
            print(
                f"Epoch {epoch + 1}: "
                f"train loss = {avg_train_loss:.4f}, "
                f"val loss = {avg_val_loss:.4f}, "
                f"perplexity = {perplexity:.2f}"
            )
    
        print(f"Saving to {args.output_dir}.")
    
        # Avoid save_pretrained error for config classes without default init values.
        if hasattr(self.model, "config"):
            self.model.config.has_no_defaults_at_init = True
    
        self.model.save_pretrained(args.output_dir)

In [49]:
from types import SimpleNamespace
from torch.utils.data import DataLoader
import torch
import numpy as np
import torch.nn.functional as F

# Re-initialize the model for full training.
# You can change embedding_size / hidden_size if you want, but these are reasonable.
config = A1RNNModelConfig(
    vocab_size=len(tokenizer),
    embedding_size=128,
    hidden_size=256
)

model = A1RNNModel(config)

# This avoids a HuggingFace save_pretrained issue with config classes
# that require arguments at initialization.
model.config.has_no_defaults_at_init = True


args = SimpleNamespace(
    output_dir="trainer_output_full",
    optim="adamw_torch",
    eval_strategy="epoch",
    learning_rate=1e-3,
    num_train_epochs=20,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    use_cpu=False,
    no_cuda=False
)


trainer = A1Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    tokenizer=tokenizer
)

trainer.train()

Device: cuda
Epoch 1: train loss = 5.1755, val loss = 4.7813, perplexity = 119.26
Epoch 2: train loss = 4.7164, val loss = 4.6194, perplexity = 101.43
Epoch 3: train loss = 4.5534, val loss = 4.5390, perplexity = 93.60
Epoch 4: train loss = 4.4848, val loss = 4.5012, perplexity = 90.13
Epoch 5: train loss = 4.4151, val loss = 4.4666, perplexity = 87.06
Epoch 6: train loss = 4.3764, val loss = 4.4418, perplexity = 84.93
Epoch 7: train loss = 4.3449, val loss = 4.4269, perplexity = 83.67
Epoch 8: train loss = 4.3403, val loss = 4.4317, perplexity = 84.07
Epoch 9: train loss = 4.3314, val loss = 4.4197, perplexity = 83.07
Epoch 10: train loss = 4.3067, val loss = 4.4076, perplexity = 82.07
Epoch 11: train loss = 4.2912, val loss = 4.4074, perplexity = 82.06
Epoch 12: train loss = 4.2936, val loss = 4.4058, perplexity = 81.92
Epoch 13: train loss = 4.2398, val loss = 4.3833, perplexity = 80.10
Epoch 14: train loss = 4.3043, val loss = 4.4095, perplexity = 82.23
Epoch 15: train loss = 4.296

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# Part 5: Evaluation and analysis
## Task 5.1: Predicting the next word

In [50]:
import torch

def predict_next_words(model, tokenizer, text, k=5):
    """
    Predict the top-k next words after a given input text.
    """
    model.eval()
    device = next(model.parameters()).device

    encoded = tokenizer(
        [text],
        return_tensors="pt",
        padding=False,
        truncation=True
    )

    input_ids = encoded["input_ids"].to(device)

    with torch.no_grad():
        output = model(input_ids=input_ids)

    logits = output.logits

    # The tokenizer adds an EOS token at the end.
    # We want to predict the word after the last real input word,
    # so we use the second-last position.
    next_word_logits = logits[0, -2, :]

    scores, indices = torch.topk(next_word_logits, k=k)

    predictions = []

    for score, idx in zip(scores, indices):
        token_id = idx.item()
        token = tokenizer.id2token[token_id]
        predictions.append((token, score.item()))

    return predictions

In [51]:
examples = [
    "She lives in San"
]

for text in examples:
    print("Input:", text)
    predictions = predict_next_words(model, tokenizer, text, k=5)

    for token, score in predictions:
        print(f"  {token:15s} {score:.4f}")

    print()

Input: She lives in San
  francisco       12.6524
  <UNK>           11.2709
  antonio         11.0141
  diego           10.7383
  salvador        9.8833



## Task 5.2: Computing the perplexity

In [52]:
from torch.utils.data import DataLoader
import torch
import numpy as np

def evaluate_perplexity(model, tokenizer, eval_dataset, batch_size=64):
    
    model.eval()
    device = next(model.parameters()).device

    loader = DataLoader(
        eval_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    loss_fct = torch.nn.CrossEntropyLoss(
        ignore_index=-100,
        reduction="sum"
    )

    total_nll = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in loader:
            encoded = tokenizer(
                batch["text"],
                return_tensors="pt",
                padding=True,
                truncation=True
            )

            input_ids = encoded["input_ids"].to(device)

            labels = input_ids.clone()
            labels[labels == tokenizer.pad_token_id] = -100

            output = model(input_ids=input_ids)
            logits = output.logits

            # Predict token t+1 from position t.
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = loss_fct(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )

            num_tokens = (shift_labels != -100).sum().item()

            total_nll += loss.item()
            total_tokens += num_tokens

    mean_nll = total_nll / total_tokens
    perplexity = np.exp(mean_nll)

    return mean_nll, perplexity

In [53]:
val_loss, val_ppl = evaluate_perplexity(
    model=model,
    tokenizer=tokenizer,
    eval_dataset=dataset["val"],
    batch_size=64
)

print("Validation loss:", val_loss)
print("Validation perplexity:", val_ppl)

Validation loss: 4.42544101506349
Validation perplexity: 83.54964575059641


## Task 5.3: Inspecting the learned word embeddings

In [54]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt

def nearest_neighbors(emb, voc, inv_voc, word, n_neighbors=5):
    if word not in voc:
        print(f"Word not in vocabulary: {word}")
        return []

    # Look up the embedding for the test word.
    test_emb = emb.weight[voc[word]]

    # Compute cosine similarity between this word and all words.
    sim_func = nn.CosineSimilarity(dim=1)
    cosine_scores = sim_func(test_emb.unsqueeze(0), emb.weight)

    # Find the positions of the highest cosine values.
    # The first one is usually the query word itself, so we skip it.
    near_nbr = cosine_scores.topk(n_neighbors + 1)

    topk_cos = near_nbr.values[1:]
    topk_indices = near_nbr.indices[1:]

    # Map indices back to strings.
    return [
        (inv_voc[ix.item()], cos.item())
        for ix, cos in zip(topk_indices, topk_cos)
    ]

In [56]:
emb = model.embedding
voc = tokenizer.token2id
inv_voc = tokenizer.id2token

test_words = ['sweden', 'denmark', 'europe', 'africa', 'london', 'stockholm', 'large', 'small', 'great', 'black', '3', '7', '10', 'seven', 'three', 'ten', '1984', '2005', '2010']

for word in test_words:
    print("Word:", word)
    neighbors = nearest_neighbors(emb, voc, inv_voc, word, n_neighbors=5)

    for neighbor, score in neighbors:
        print(f"  {neighbor:15s} {score:.4f}")

    print()

Word: sweden
  persia          0.3626
  greece          0.3550
  france          0.3523
  malaysia        0.3497
  switzerland     0.3479

Word: denmark
  belarus         0.5149
  bangladesh      0.4221
  denver          0.4206
  rica            0.4100
  ecuador         0.3977

Word: europe
  albania         0.4170
  spain           0.3947
  persia          0.3889
  canada          0.3875
  asia            0.3544

Word: africa
  country         0.3933
  iceland         0.3811
  bangladesh      0.3777
  baghdad         0.3766
  plains          0.3468

Word: london
  lincoln         0.3499
  edinburgh       0.3404
  1924            0.3336
  sweden          0.3253
  bethlehem       0.3162

Word: stockholm
  norway          0.4480
  1983            0.3434
  1864            0.3180
  1938            0.3152
  moines          0.3136

Word: large
  larger          0.5003
  smaller         0.4792
  small           0.4724
  vast            0.4059
  substantial     0.4044

Word: small
  smaller   